In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
sys.path.insert(0, project_root)

In [2]:
import logging
import os

from dotenv import load_dotenv
from opensearchpy import OpenSearch

from src.retrieval.embedder import Embedder
from src.retrieval.searcher import HybridSearcher


In [3]:
load_dotenv()

True

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [5]:
client = OpenSearch(
    hosts=[{
        "host": os.getenv("OPENSEARCH_HOST", "localhost"),
        "port": int(os.getenv("OPENSEARCH_PORT", 9200))
    }],
    http_compress=True
)

embedder = Embedder(ollama_url=os.getenv("OLLAMA_URL"))
searcher = HybridSearcher(client=client, embedder=embedder, top_k=5)

In [6]:
query = "Qual o tratamento para crise aguda de angioedema hereditário?"
results = searcher.search(query)

print(f"\nQuery: {query}")
print(f"{'='*60}")
for i, result in enumerate(results):
    print(f"\n[{i+1}] Score: {result.score:.4f}")
    print(f"     Source: {result.source}")
    print(f"     Tokens: {result.metadata['token_count']}")
    print(f"     Text: {result.text[:200]}")

2026-07-29 23:34:55,070 - src.retrieval.searcher - INFO - Searching for: 'Qual o tratamento para crise aguda de angioedema hereditário?'
2026-07-29 23:34:55,832 - opensearch - INFO - POST http://localhost:9200/angioedema/_search [status:200 request:0.084s]
2026-07-29 23:34:55,858 - opensearch - INFO - POST http://localhost:9200/angioedema/_search [status:200 request:0.026s]
2026-07-29 23:34:55,858 - src.retrieval.searcher - INFO - Semantic hits: 5 | Keyword hits: 5
2026-07-29 23:34:55,858 - src.retrieval.searcher - INFO - Returning 5 results after RRF fusion



Query: Qual o tratamento para crise aguda de angioedema hereditário?

[1] Score: 0.0167
     Source: ASBAI - O que é Angioedema.pdf
     Tokens: 510
     Text: .br/pacientes.php
73 | Doutor, eu tenho Angioedema Hereditário

Doutor,
existe um dia para o AEH?
im. O dia 16 de maio é o “Dia Mundial
de Conscientização do Angioedema
Hereditário”.
74 | Doutor, eu t

[2] Score: 0.0167
     Source: ANVISA - Icatibanto.pdf
     Tokens: 645
     Text: a análise
tenha refletido, de forma objetiva, tanto o cuidado atual oferecido aos pacientes, quando os
benefícios tangíveis de icatibanto, um medicamento eficaz e seguro para o tratamento de crises
de

[3] Score: 0.0164
     Source: ASBAI - O que é Angioedema.pdf
     Tokens: 597
     Text: promovendo atividades direcionadas para o Angioedema
Hereditário. Realiza projetos educativos de esclarecimento
sobre a doença, capacitando profissionais e Serviços de Re-
ferência no Brasil.
A página

[4] Score: 0.0164
     Source: ANVISA - Icatibanto.pdf
     

In [ ]:
from src.retrieval.reranker import Reranker

reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=5)

# busca com top_k maior para dar mais candidatos ao reranker
searcher_wide = HybridSearcher(client=client, embedder=embedder, top_k=20)

query = "Qual o tratamento para crise aguda de angioedema hereditário?"

results = searcher_wide.search(query)
reranked = reranker.rerank(query, results)

print(f"\nQuery: {query}")
print(f"{'='*60}")
for i, result in enumerate(reranked):
    print(f"\n[{i+1}] Score: {result.score:.4f}")
    print(f"     Source: {result.source}")
    print(f"     Tokens: {result.metadata['token_count']}")
    print(f"     Text: {result.text[:300]}")